In [1]:
import json 
import numpy as np 
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, GlobalAveragePooling1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
import pickle

with open('intents1.json') as file:
    data = json.load(file)
    
training_sentences = []
training_labels = []
labels = []
responses = []

for intent in data['intents']:
    for pattern in intent['patterns']:
        training_sentences.append(pattern)
        training_labels.append(intent['tag'])
    responses.append(intent['responses'])
    
    if intent['tag'] not in labels:
        labels.append(intent['tag'])
        
num_classes = len(labels)

lbl_encoder = LabelEncoder()
lbl_encoder.fit(training_labels)
training_labels = lbl_encoder.transform(training_labels)

vocab_size = 1000
embedding_dim = 16
max_len = 20
oov_token = "<OOV>"

tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_token)
tokenizer.fit_on_texts(training_sentences)
word_index = tokenizer.word_index
sequences = tokenizer.texts_to_sequences(training_sentences)
padded_sequences = pad_sequences(sequences, truncating='post', maxlen=max_len)

# Model Definition
model = Sequential()
model.add(Embedding(vocab_size, embedding_dim, input_length=max_len))
model.add(GlobalAveragePooling1D())
model.add(Dense(16, activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(num_classes, activation='softmax'))

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train (Increase epochs if the bot is still "dumb")
model.fit(padded_sequences, np.array(training_labels), epochs=500, verbose=0)

# SAVE EVERYTHING
model.save("chat_model.keras")

with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)
    
with open('label_encoder.pickle', 'wb') as ecn_file:
    pickle.dump(lbl_encoder, ecn_file, protocol=pickle.HIGHEST_PROTOCOL)

print("Training Complete. Files saved.")

C:\Users\chauh\miniconda3\envs\codexenv\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Training Complete. Files saved.


In [ ]:
import json
import numpy as np
from tensorflow import keras
import colorama
import pickle
from colorama import Fore, Style, init

init(autoreset=True)

with open("intents1.json") as file:
    data = json.load(file)

def chat():
    # Load all files
    model = keras.models.load_model('chat_model.keras')
    
    with open('tokenizer.pickle', 'rb') as handle:
        tokenizer = pickle.load(handle)

    with open('label_encoder.pickle', 'rb') as enc:
        lbl_encoder = pickle.load(enc)

    max_len = 20

    print(Fore.YELLOW + "Start messaging with the bot (type quit to stop)!" + Style.RESET_ALL)

    while True:
        print(Fore.LIGHTBLUE_EX + "User: " + Style.RESET_ALL, end="")
        inp = input()
        if inp.lower() == "quit":
            break

        # Convert user input to sequence
        seq = tokenizer.texts_to_sequences([inp])
        padded = keras.preprocessing.sequence.pad_sequences(seq, truncating='post', maxlen=max_len)
        
        # Predict
        result = model.predict(padded, verbose=0)
        
        # Get the predicted tag string
        # FIX: We use [0] because inverse_transform returns an array
        tag = lbl_encoder.inverse_transform([np.argmax(result)])[0]

        # Find response in JSON
        found_response = False
        for i in data['intents']:
            if i['tag'] == tag:
                print(Fore.GREEN + "ChatBot: " + Style.RESET_ALL, np.random.choice(i['responses']))
                found_response = True
        
        if not found_response:
             print(Fore.RED + "ChatBot: I'm sorry, I don't understand that.")

chat()

In [6]:
import streamlit as st
import json
import numpy as np
import pickle
from tensorflow import keras

# Page Configuration
st.set_page_config(page_title="AI Chatbot", page_icon="🤖")
st.title("My AI Assistant")

# --- LOAD ASSETS (Cached so they only load once) ---
@st.cache_resource
def load_chat_assets():
    with open("intents1.json") as file:
        data = json.load(file)
    model = keras.models.load_model('chat_model.keras')
    with open('tokenizer.pickle', 'rb') as handle:
        tokenizer = pickle.load(handle)
    with open('label_encoder.pickle', 'rb') as enc:
        lbl_encoder = pickle.load(enc)
    return data, model, tokenizer, lbl_encoder

data, model, tokenizer, lbl_encoder = load_chat_assets()

# --- CHAT HISTORY INITIALIZATION ---
if "messages" not in st.session_state:
    st.session_state.messages = []

# Display chat messages from history on app rerun
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# --- CHAT LOGIC ---
if prompt := st.chat_input("Type your message here..."):
    # Display user message
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    # Predict response
    max_len = 20
    seq = tokenizer.texts_to_sequences([prompt])
    padded = keras.preprocessing.sequence.pad_sequences(seq, truncating='post', maxlen=max_len)
    result = model.predict(padded, verbose=0)
    
    tag = lbl_encoder.inverse_transform([np.argmax(result)])[0]
    
    # Get random response from JSON
    response = "I'm sorry, I don't understand."
    for i in data['intents']:
        if i['tag'] == tag:
            response = np.random.choice(i['responses'])

    # Display assistant response
    with st.chat_message("assistant"):
        st.markdown(response)
    st.session_state.messages.append({"role": "assistant", "content": response})

2026-02-18 10:35:00.506 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-18 10:35:00.509 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-18 10:35:02.292 
  command:

    streamlit run C:\Users\chauh\miniconda3\envs\codexenv\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-02-18 10:35:02.296 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-18 10:35:02.298 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-18 10:35:02.303 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-18 10:35:02.395 Thread 'MainThread': missing ScriptRunContext! This warning can be ig

In [ ]:
%%writefile app.py
import streamlit as st
import json
import numpy as np
import pickle
from tensorflow import keras

st.title("Jupyter-Hosted Chatbot")

# Your existing chat logic here...
# (Paste the full Streamlit code I gave you in the previous response here)


# Page Configuration
st.set_page_config(page_title="AI Chatbot", page_icon="🤖")
st.title("My AI Assistant")

# --- LOAD ASSETS (Cached so they only load once) ---
@st.cache_resource
def load_chat_assets():
    with open("intents.json") as file:
        data = json.load(file)
    model = keras.models.load_model('chat_model.keras')
    with open('tokenizer.pickle', 'rb') as handle:
        tokenizer = pickle.load(handle)
    with open('label_encoder.pickle', 'rb') as enc:
        lbl_encoder = pickle.load(enc)
    return data, model, tokenizer, lbl_encoder

data, model, tokenizer, lbl_encoder = load_chat_assets()

# --- CHAT HISTORY INITIALIZATION ---
if "messages" not in st.session_state:
    st.session_state.messages = []

# Display chat messages from history on app rerun
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# --- CHAT LOGIC ---
if prompt := st.chat_input("Type your message here..."):
    # Display user message
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    # Predict response
    max_len = 20
    seq = tokenizer.texts_to_sequences([prompt])
    padded = keras.preprocessing.sequence.pad_sequences(seq, truncating='post', maxlen=max_len)
    result = model.predict(padded, verbose=0)
    
    tag = lbl_encoder.inverse_transform([np.argmax(result)])[0]
    
    # Get random response from JSON
    response = "I'm sorry, I don't understand."
    for i in data['intents']:
        if i['tag'] == tag:
            response = np.random.choice(i['responses'])

    # Display assistant response
    with st.chat_message("assistant"):
        st.markdown(response)
    st.session_state.messages.append({"role": "assistant", "content": response})

In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

In [2]:
import json 
import numpy as np 
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, GlobalAveragePooling1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
import pickle

# Load the intents file
with open('intents1.json') as file:
    data = json.load(file)
    
training_sentences = []
training_labels = []
labels = []

for intent in data['intents']:
    for pattern in intent['patterns']:
        training_sentences.append(pattern)
        training_labels.append(intent['tag'])
    
    if intent['tag'] not in labels:
        labels.append(intent['tag'])
        
num_classes = len(labels)

# Convert text labels to numbers
lbl_encoder = LabelEncoder()
lbl_encoder.fit(training_labels)
training_labels = lbl_encoder.transform(training_labels)

# Text Preprocessing
vocab_size = 1000
embedding_dim = 16
max_len = 20
oov_token = "<OOV>"

tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_token)
tokenizer.fit_on_texts(training_sentences)
sequences = tokenizer.texts_to_sequences(training_sentences)
padded_sequences = pad_sequences(sequences, truncating='post', maxlen=max_len)

# Build the Neural Network
model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=max_len),
    GlobalAveragePooling1D(),
    Dense(16, activation='relu'),
    Dense(16, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train the model
print("Training ChatBot's brain...")
model.fit(padded_sequences, np.array(training_labels), epochs=500, verbose=0)

# Save all necessary files
model.save("chat_model.keras")
with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle)
with open('label_encoder.pickle', 'wb') as ecn_file:
    pickle.dump(lbl_encoder, ecn_file)

print("Training Complete! ChatBot is ready to talk.")

Training ChatBot's brain...


C:\Users\chauh\miniconda3\envs\codexenv\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Training Complete! ChatBot is ready to talk.
